In [1]:
import set_env

# {class}`~torch.nn.PixelShuffle`

In [ ]:
import torch
from tvm import relay
import tvm

In [9]:
input_shape = [1, 144, 16, 16]
torch.set_grad_enabled(False)
model = torch.nn.PixelShuffle(2).float().eval()
input_shapes = [(f"x_{index}", shape) for index, shape in enumerate(input_shape)]
trace = torch.jit.trace(model, [torch.rand(input_shape).float()])
mod, params = relay.frontend.from_pytorch(
    trace,
    input_shapes,
)
target = "llvm"
dev = tvm.device(target, 0)
exe = relay.create_executor(
    "vm", mod=mod, params=params, device=dev, target=target
).evaluate()
input_names = [f"x_{idx}" for idx, _ in enumerate(inputs)]
compiled_input = dict(zip(input_names, [inp.clone().cpu().numpy() for inp in inputs]))
result = exe(**compiled_input)

RuntimeError: PyTorch has 1 inputs and input_infos lists 4.

In [7]:
result

<tvm.nd.NDArray shape=(1, 36, 32, 32), cpu(0)>
array([[[[0.45167208, 0.4751528 , 0.66562855, ..., 0.10102624,
          0.47107768, 0.98546124],
         [0.50281024, 0.9156416 , 0.97454673, ..., 0.33834815,
          0.99482936, 0.1813637 ],
         [0.36587209, 0.77844733, 0.9402926 , ..., 0.07727742,
          0.5027055 , 0.03256208],
         ...,
         [0.6315825 , 0.32724112, 0.1491878 , ..., 0.38751185,
          0.71438605, 0.8736542 ],
         [0.23701996, 0.93006486, 0.79464227, ..., 0.74860734,
          0.10734552, 0.49250185],
         [0.30654496, 0.90148944, 0.536291  , ..., 0.4301917 ,
          0.1279993 , 0.517763  ]],

        [[0.70649207, 0.01436931, 0.9452317 , ..., 0.50734496,
          0.15042186, 0.75830054],
         [0.81990063, 0.32498264, 0.7660303 , ..., 0.13331008,
          0.12531197, 0.8993785 ],
         [0.84151685, 0.15428996, 0.74632305, ..., 0.0347752 ,
          0.35647523, 0.41170758],
         ...,
         [0.21683067, 0.25745237, 0.31181